# BERT Baseline (Colab)

Fine-tuned `bert-base-uncased` sentence classifier with:
- **same split** as `bert_attention_pipeline.ipynb` (loaded from `bert_bias_classifier_v9_split.npz`)
- train_fit / val / test partition (val for early stopping)
- test set restricted to real/original sentences
- multi-seed evaluation (different weight init, same data split)
- optional LOSO
- checkpoint save/load

In [ ]:
!pip -q install -U transformers torch scikit-learn pandas numpy

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

In [ ]:
def find_root(start: Path, repo_name: str = 'project') -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if candidate.name == repo_name:
            return candidate
    return start


_nb_file = Path(globals().get('__vsc_ipynb_file__', '') or '')
if _nb_file.exists():
    root_dir = find_root(_nb_file.parent)
else:
    root_dir = find_root(Path.cwd())

notebook_dir = root_dir / 'dataset' / 'v2'

DATA_JSON = notebook_dir / 'bias_sentences_v9.json'
FEAT_PKL = notebook_dir / 'feature_matrix_bert_v9.pkl'
SPLIT_NPZ = root_dir / 'attention_app' / 'bias' / 'models' / 'bert_bias_classifier_v9_split.npz'

OUT_DIR = notebook_dir / 'bert_baseline_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Root dir :', root_dir)
print('DATA_JSON:', DATA_JSON)
print('FEAT_PKL :', FEAT_PKL)
print('SPLIT_NPZ:', SPLIT_NPZ)
print('OUT_DIR  :', OUT_DIR)

In [ ]:
with open(DATA_JSON, encoding='utf-8') as f:
    raw = json.load(f)

df_sentences = pd.DataFrame(raw['entries']).copy()
df_sentences['label'] = df_sentences['has_bias'].astype(int)

SOURCE_CANONICAL = {
    'biased_corpus_only': 'biased-corpus',
    'biased_corpus_v2': 'biased-corpus',
    'gemini_only': 'gemini',
    'gemini_only_v2': 'gemini',
    'gus_only': 'gus-dataset',
    'gus_only_v2': 'gus-dataset',
}
df_sentences['source_canonical'] = (
    df_sentences['source'].map(SOURCE_CANONICAL).fillna(df_sentences['source'])
)

df_features = pd.read_pickle(FEAT_PKL).copy()

if len(df_features) != len(df_sentences):
    raise RuntimeError(f'Size mismatch: features={len(df_features)} vs json={len(df_sentences)}')

for col in [
    'text', 'source', 'source_canonical', 'original_id', 'role', 'topic',
    'pair_id', 'sentence_id', 'edit_type',
]:
    if col in df_sentences.columns:
        df_features[col] = df_sentences[col].values

y = df_features['label'].astype(int)
texts = df_features['text'].values
sources = df_features['source_canonical'].values
unique_sources = np.array(sorted(np.unique(sources)))

# ── Load persisted split (same as bert_attention_pipeline.ipynb) ──
split = np.load(SPLIT_NPZ)
train_idx = split['train_idx']
test_idx = split['test_idx']
train_fit_idx = split['train_fit_idx']
val_idx = split['val_idx']

print('N:', len(df_features))
print('Label dist:', y.value_counts().to_dict())
print('Sources:', pd.Series(sources).value_counts().to_dict())
print(f'\nLoaded split from {SPLIT_NPZ.name}:')
print(f'  train_fit: {len(train_fit_idx)}  val: {len(val_idx)}  test: {len(test_idx)}')
print(f'  train (fit+val): {len(train_idx)}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

BASELINE_SEEDS = [1, 2, 3, 4, 5]  # set [1,2,3] if you need lighter runs
MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 20
LR = 2e-5

RUN_LOSO = True
USE_SAVED_MODELS = True
SAVE_MODELS = True

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def encode_texts(text_array, tok, max_len):
    enc = tok(
        list(text_array),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt',
    )
    return enc['input_ids'], enc['attention_mask']


def train_bert_classifier(X_ids, X_mask, y_train,
                          X_ids_val, X_mask_val, y_val,
                          seed, epochs=EPOCHS, lr=LR, patience=3):
    """Train with validation-based early stopping."""
    set_all_seeds(seed)
    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased', num_labels=2
    ).to(device)

    dataset = TensorDataset(
        X_ids.to(device),
        X_mask.to(device),
        torch.tensor(y_train, dtype=torch.long).to(device),
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(loader) * epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    best_val_f1 = -1.0
    best_state = None
    wait = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for ids, mask, labels in loader:
            optimizer.zero_grad()
            outputs = model(input_ids=ids, attention_mask=mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # Validation
        val_probs = predict_bert(model, X_ids_val, X_mask_val)
        val_preds = (val_probs >= 0.5).astype(int)
        from sklearn.metrics import f1_score as _f1
        val_f1 = _f1(y_val, val_preds, zero_division=0)

        print(f'    Epoch {epoch+1}/{epochs}  loss={total_loss/len(loader):.4f}  val_f1={val_f1:.4f}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'    Early stopping at epoch {epoch+1} (best val_f1={best_val_f1:.4f})')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
    return model


def predict_bert(model, X_ids, X_mask):
    model.eval()
    dataset = TensorDataset(X_ids.to(device), X_mask.to(device))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE)
    all_probs = []
    with torch.no_grad():
        for ids, mask in loader:
            outputs = model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1].cpu().numpy()
            all_probs.append(probs)
    return np.concatenate(all_probs)


def ckpt_main(seed):
    return OUT_DIR / f'bert_seed_{seed}'


def ckpt_loso(seed, src):
    safe_src = str(src).replace('/', '_')
    return OUT_DIR / f'bert_seed_{seed}_loso_{safe_src}'

In [ ]:
bert_ft_seed_results = []
bert_ft_seed_loso = []

# Encode val and test once (same across all seeds)
ids_val, mask_val = encode_texts(texts[val_idx], tokenizer, MAX_LEN)
ids_te, mask_te = encode_texts(texts[test_idx], tokenizer, MAX_LEN)
y_val_arr = y.iloc[val_idx].values
y_te = y.iloc[test_idx].values

# Encode train_fit once
ids_tr, mask_tr = encode_texts(texts[train_fit_idx], tokenizer, MAX_LEN)
y_tr = y.iloc[train_fit_idx].values

for seed in BASELINE_SEEDS:
    print('\n' + '='*60)
    print(f'Fine-tuned BERT - SEED = {seed}')
    print('='*60)

    main_dir = ckpt_main(seed)
    if USE_SAVED_MODELS and main_dir.exists():
        print(f'  Loading checkpoint: {main_dir}')
        model = BertForSequenceClassification.from_pretrained(main_dir).to(device)
    else:
        model = train_bert_classifier(
            ids_tr, mask_tr, y_tr,
            ids_val, mask_val, y_val_arr,
            seed,
        )
        if SAVE_MODELS:
            main_dir.mkdir(parents=True, exist_ok=True)
            model.save_pretrained(main_dir)
            tokenizer.save_pretrained(main_dir)
            print(f'  Saved checkpoint: {main_dir}')

    probs_te = predict_bert(model, ids_te, mask_te)
    preds_te = (probs_te >= 0.5).astype(int)

    acc_b = accuracy_score(y_te, preds_te)
    f1_b = f1_score(y_te, preds_te, zero_division=0)
    auc_b = roc_auc_score(y_te, probs_te)

    # Val metrics
    probs_val = predict_bert(model, ids_val, mask_val)
    preds_val = (probs_val >= 0.5).astype(int)
    f1_val = f1_score(y_val_arr, preds_val, zero_division=0)

    print(f'  Train-fit={len(train_fit_idx)}  Val={len(val_idx)}  Test={len(test_idx)}')
    print(f'  Val  F1={f1_val:.4f}')
    print(f'  Test Acc={acc_b:.4f}  F1={f1_b:.4f}  AUC={auc_b:.4f}')

    loso_mean_f1 = np.nan
    if RUN_LOSO:
        loso_f1s = []
        for test_src in unique_sources:
            tmask = sources == test_src
            tr_l = np.where(~tmask)[0]
            te_l = np.where(tmask)[0]

            # LOSO val: 20% of train
            from sklearn.model_selection import GroupShuffleSplit
            loso_pairs = df_features.iloc[tr_l]['pair_id'].copy()
            loso_na = loso_pairs.isna()
            loso_pairs[loso_na] = [f'unpaired_{i}' for i in range(loso_na.sum())]
            gss_lv = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
            tr_lv, val_lv = next(gss_lv.split(tr_l, y.iloc[tr_l], groups=loso_pairs.values))
            tr_l_fit = tr_l[tr_lv]
            val_l = tr_l[val_lv]

            ids_tr_l, mask_tr_l = encode_texts(texts[tr_l_fit], tokenizer, MAX_LEN)
            ids_val_l, mask_val_l = encode_texts(texts[val_l], tokenizer, MAX_LEN)
            ids_te_l, mask_te_l = encode_texts(texts[te_l], tokenizer, MAX_LEN)

            loso_dir = ckpt_loso(seed, test_src)
            if USE_SAVED_MODELS and loso_dir.exists():
                print(f'    [LOSO {test_src}] loading: {loso_dir}')
                model_l = BertForSequenceClassification.from_pretrained(loso_dir).to(device)
            else:
                model_l = train_bert_classifier(
                    ids_tr_l, mask_tr_l, y.iloc[tr_l_fit].values,
                    ids_val_l, mask_val_l, y.iloc[val_l].values,
                    seed,
                )
                if SAVE_MODELS:
                    loso_dir.mkdir(parents=True, exist_ok=True)
                    model_l.save_pretrained(loso_dir)
                    tokenizer.save_pretrained(loso_dir)
                    print(f'    [LOSO {test_src}] saved: {loso_dir}')

            probs_l = predict_bert(model_l, ids_te_l, mask_te_l)
            preds_l = (probs_l >= 0.5).astype(int)
            f1_l = f1_score(y.iloc[te_l].values, preds_l, zero_division=0)
            loso_f1s.append(f1_l)
            bert_ft_seed_loso.append({'seed': seed, 'source': test_src, 'f1': f1_l})

            del model_l
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        loso_mean_f1 = float(np.mean(loso_f1s))
        print(f'  LOSO mean F1={loso_mean_f1:.4f}')

    bert_ft_seed_results.append({
        'seed': seed,
        'accuracy': acc_b,
        'f1': f1_b,
        'auc': auc_b,
        'val_f1': f1_val,
        'loso_mean_f1': loso_mean_f1,
    })

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

df_bert_ft = pd.DataFrame(bert_ft_seed_results)
df_bert_loso = pd.DataFrame(bert_ft_seed_loso)

display(df_bert_ft)
if not df_bert_loso.empty:
    display(df_bert_loso.head())

In [ ]:
print('\n' + '='*60)
print(f'Fine-tuned BERT SUMMARY - {len(BASELINE_SEEDS)} seeds')
print('='*60)

for col in ['accuracy', 'f1', 'auc', 'val_f1', 'loso_mean_f1']:
    if col in df_bert_ft.columns and df_bert_ft[col].notna().any():
        m, s = df_bert_ft[col].mean(), df_bert_ft[col].std()
        print(f'  {col:15s}: {m:.4f} +/- {s:.4f}')

df_bert_ft.to_csv(OUT_DIR / 'df_bert_ft.csv', index=False)
if not df_bert_loso.empty:
    df_bert_loso.to_csv(OUT_DIR / 'df_bert_loso.csv', index=False)

print('\nSaved:')
print(' -', OUT_DIR / 'df_bert_ft.csv')
if not df_bert_loso.empty:
    print(' -', OUT_DIR / 'df_bert_loso.csv')